# Python collections

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-02-collections.ipynb)

Almost every pipeline moves records between collections: a list of rows, a dictionary keyed by customer, a set of IDs
already seen. This chapter covers the four built-in collections, how to group and count with them, and the mutability
rules that cause the most subtle bugs.

## Lists: ordered and changeable

A list keeps order and allows duplicates. You can add, change and remove items in place. Indexes start at 0, and
negative indexes count from the end.

`append` adds one item, `extend` adds every item of another iterable, and `+` builds a new list.

### Where you'll use it

- **Batching records:** collect rows into a list until it reaches a size, then write it in one call to the database or API.
- **Ordered output:** keep the order of events when a file is replayed, because downstream logic often depends on it.

In [ ]:
rows = ["a", "b", "c"]
assert rows[0] == "a" and rows[-1] == "c"
assert rows[1:] == ["b", "c"]

rows.append("d")
assert rows == ["a", "b", "c", "d"]

rows.extend(["e", "f"])
assert len(rows) == 6

# append adds ONE item even if that item is a list.
nested = [1, 2]
nested.append([3, 4])
assert len(nested) == 3

## Sorting

`sorted()` returns a new list. `list.sort()` sorts in place and returns `None`, which is a common bug:
`x = items.sort()` leaves `x` as `None`.

Use `key=` to sort by a field. Sorting is stable, so equal keys keep their original order.

### Where you'll use it

- **Choosing the latest record:** sort by `(business_key, updated_at)` and keep the last row per key.
- **Deterministic output:** sorting before writing files makes diffs and tests stable, so reruns produce byte-identical outputs.

In [ ]:
orders = [
    {"id": 1, "amount": 50, "day": "mon"},
    {"id": 2, "amount": 20, "day": "tue"},
    {"id": 3, "amount": 50, "day": "wed"},
]

by_amount = sorted(orders, key=lambda o: o["amount"])
assert [o["id"] for o in by_amount] == [2, 1, 3]        # stable: 1 stays before 3

biggest_first = sorted(orders, key=lambda o: (-o["amount"], o["id"]))
assert [o["id"] for o in biggest_first] == [1, 3, 2]

result = sorted(orders, key=lambda o: o["amount"]).copy()   # sorted() already returns a new list
assert result is not orders

## Tuples: fixed records

A tuple is an immutable sequence. Use it for a record whose shape never changes, such as a `(key, value)` pair or a row
returned from a query. **Unpacking** assigns each part to a name. Swapping two values needs no temporary variable.

### Where you'll use it

- **Database rows:** DB-API drivers return rows as tuples. Unpacking them into names is the everyday pattern.
- **Composite keys:** `(customer_id, order_date)` as a dictionary key for deduplication or a lookup table.

In [ ]:
key, value = ("customer_42", 180.5)
assert key == "customer_42" and value == 180.5

a, b = 1, 2
a, b = b, a
assert (a, b) == (2, 1)

# Star unpacking takes the rest.
first, *middle, last = [10, 20, 30, 40]
assert first == 10 and middle == [20, 30] and last == 40

try:
    key_tuple = ("x", 1)
    key_tuple[0] = "y"
except TypeError:
    pass
else:
    raise AssertionError("tuples cannot be changed")

## Sets: unique items and fast membership

A set holds unique items with no order. Checking `in` is fast, which makes sets the right tool for "have I seen this
ID?". Set algebra gives you union, intersection and difference.

### Where you'll use it

- **Deduplication:** keep the set of IDs already processed so a replayed message is skipped.
- **Change detection:** the difference between yesterday's and today's keys gives the inserts and deletes for a snapshot diff.
- **Access control:** checking whether a user's role set contains a required role.

In [ ]:
seen = set()
duplicates = []
for event_id in ["e1", "e2", "e1", "e3", "e2"]:
    if event_id in seen:
        duplicates.append(event_id)
    seen.add(event_id)

assert duplicates == ["e1", "e2"]
assert seen == {"e1", "e2", "e3"}

yesterday = {"a", "b", "c"}
today = {"b", "c", "d"}
assert yesterday & today == {"b", "c"}        # both days
assert yesterday | today == {"a", "b", "c", "d"}
assert yesterday - today == {"a"}              # gone today
assert today - yesterday == {"d"}              # new today

## Dictionaries: keyed lookup

A dictionary maps keys to values. Lookups by key are fast. Three methods save most code:

- `d.get(key, default)` returns a default instead of raising `KeyError`.
- `d.setdefault(key, [])` returns the existing value or stores a default.
- `d.items()` iterates key and value pairs.

Since Python 3.7 dictionaries keep insertion order.

### Where you'll use it

- **Enrichment joins in Python:** a dictionary built from a small reference table, used to look up names for each event.
- **Config and settings:** most pipeline configuration is a dictionary loaded from YAML or JSON.

In [ ]:
stock = {"apples": 4}
assert stock.get("pears", 0) == 0
assert stock.get("apples") == 4

stock["pears"] = stock.get("pears", 0) + 3
assert stock == {"apples": 4, "pears": 3}

assert list(stock) == ["apples", "pears"]     # insertion order
assert list(stock.items())[0] == ("apples", 4)

# Merge two dicts; the right-hand side wins on conflicts.
merged = {**{"a": 1, "b": 2}, **{"b": 20, "c": 3}}
assert merged == {"a": 1, "b": 20, "c": 3}

## Grouping and counting

Two patterns come up constantly in pipelines:

- **Group rows by key.** `collections.defaultdict(list)` creates the empty list the first time a key appears.
- **Count by key.** `collections.Counter` counts items. `most_common(n)` gives the top `n`.

These are the Python versions of `GROUP BY key` and `COUNT(*)`.

### Where you'll use it

- **Aggregation before the warehouse:** counting events per user or per status when a full SQL engine is too heavy for a small file.
- **Data profiling:** `Counter` on a column shows the distinct values and their frequency, which is the first step of a data-quality check.

In [ ]:
from collections import Counter, defaultdict

events = [("ana", "click"), ("ben", "view"), ("ana", "view"), ("ana", "click")]

by_user = defaultdict(list)
for user, action in events:
    by_user[user].append(action)

assert by_user["ana"] == ["click", "view", "click"]
assert by_user["zed"] == []                    # reading a missing key creates it

counts = Counter(action for _, action in events)
assert counts == Counter({"click": 2, "view": 2})
assert counts.most_common(1) == [("click", 2)]

## Dictionary comprehensions and nested data

Comprehensions build collections in one line. Use them when they stay readable. A dictionary comprehension builds a
mapping from any iterable.

### Where you'll use it

- **Flattening API payloads:** nested JSON reads as `order["customer"]["address"]["city"]`. Safe lookups prevent one missing field from failing the batch.
- **Renaming columns:** a dictionary maps source field names to warehouse column names.

In [ ]:
prices = {"apple": 0.5, "pear": 0.8, "fig": 2.0}

cheap = {item: price for item, price in prices.items() if price < 1.0}
assert cheap == {"apple": 0.5, "pear": 0.8}

pairs = {name: len(name) for name in ["ana", "ben"]}
assert pairs == {"ana": 3, "ben": 3}

# Nested data: read the path one key at a time, with a default at each level.
order = {"customer": {"address": {"city": "Oslo"}}}
city = order.get("customer", {}).get("address", {}).get("city", "unknown")
assert city == "Oslo"
missing = {}.get("customer", {}).get("address", {}).get("city", "unknown")
assert missing == "unknown"

## Mutability and copies

Assignment copies a reference, not the data. Two names can point at one list, so a change through one name shows
through the other. A **shallow copy** (`list(x)`, `x.copy()`, `dict(x)`) copies the outer container only. Nested
objects are still shared. `copy.deepcopy` copies everything.

### Where you'll use it

- **Avoiding shared-state bugs:** a shared config dictionary mutated by one job changes the next job's settings unless it is copied.
- **Snapshots:** keep a deep copy of a record before a transformation, so you can diff the input and output in tests.

In [ ]:
import copy

a = [1, 2, 3]
b = a                     # same list, two names
b.append(4)
assert a == [1, 2, 3, 4]

c = list(a)               # a new outer list
c.append(5)
assert a == [1, 2, 3, 4]

grid = [[0, 0], [0, 0]]
shallow = list(grid)
shallow[0][0] = 9         # the inner lists are still shared
assert grid[0][0] == 9

deep = copy.deepcopy(grid)
deep[1][1] = 7
assert grid[1][1] == 0

## Practice questions

1. Given a list of `(customer, amount)` pairs, return the total per customer as a dictionary.
2. Return the IDs that appear in both `yesterday` and `today`, sorted, from two lists (duplicates may exist).
3. Remove duplicates from a list while keeping the first occurrence of each item and the original order.

## Answers

### Question 1

In [ ]:
def totals_by_customer(pairs):
    totals = {}
    for customer, amount in pairs:
        totals[customer] = totals.get(customer, 0) + amount
    return totals

assert totals_by_customer([("a", 5), ("b", 2), ("a", 3)]) == {"a": 8, "b": 2}

### Question 2

In [ ]:
def in_both(yesterday, today):
    return sorted(set(yesterday) & set(today))

assert in_both([3, 1, 1, 2], [2, 2, 3, 9]) == [2, 3]

### Question 3

In [ ]:
def dedupe_keep_order(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

assert dedupe_keep_order([3, 1, 3, 2, 1]) == [3, 1, 2]

## Interview notes

- **`list.sort()` returns `None`.** Use `sorted()` when you need a value.
- **Sets give O(1) average membership.** Scanning a list for membership is O(n) per check, so a set turns a quadratic loop
  into a linear one.
- **Use `defaultdict` or `Counter` for grouping.** A hand-written `if key not in d` is correct but noisier.
- **Shallow copies share nested objects.** Mutating a nested list through a shallow copy changes the original.